# YOLOv11 Training Notebook
## Real-time Student Behavior Detection (FYP)

This notebook is rewritten in correct execution order. Run cells from top to bottom.

## 1. Import Required Libraries and Setup

In [1]:
import sys, platform
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Platform: Linux-6.6.113+-x86_64-with-glibc2.35


In [2]:
%pip install -q ultralytics opencv-python pyyaml matplotlib pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s eta 0:00:0000:01


In [3]:
from pathlib import Path
import json
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# Optional: mount Google Drive when running in Colab
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Prefer project-local relative paths: dataset/...
if Path('dataset/data.yaml').exists():
    PROJECT_ROOT = Path('.')
    DATASET_ROOT = Path('dataset')
elif Path('/content/dataset/data.yaml').exists():
    PROJECT_ROOT = Path('/content')
    DATASET_ROOT = PROJECT_ROOT / 'dataset'
elif Path('/content/drive/MyDrive/datasets/data.yaml').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive')
    DATASET_ROOT = PROJECT_ROOT / 'datasets'
elif Path('/content/drive/MyDrive/dataset/data.yaml').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive')
    DATASET_ROOT = PROJECT_ROOT / 'dataset'
else:
    PROJECT_ROOT = Path('.')
    DATASET_ROOT = Path('dataset')

DATA_YAML = DATASET_ROOT / 'data.yaml'
RUN_DIR = Path('fyp_runs') / 'classroom_model_v1'
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'

# Device selection (CUDA on NVIDIA; ROCm can also appear through torch.cuda)
device = '0' if torch.cuda.is_available() else 'cpu'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Data yaml: {DATA_YAML}')
print(f'PyTorch version: {torch.__version__}')
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')
print(f'Training device setting: {device}')

if not DATA_YAML.exists():
    print('Dataset not found. Expected relative path: dataset/data.yaml')
    print('Create this structure in your runtime:')
    print('dataset/train/images, dataset/train/labels, dataset/valid/images, dataset/valid/labels, dataset/test/images, dataset/test/labels')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Project root: .
Dataset root: dataset
Data yaml: dataset/data.yaml
PyTorch version: 2.10.0+cu128
torch.cuda.is_available(): True
Device name: Tesla T4
Training device setting: 0
Dataset not found. Expected relative path: dataset/data.yaml
Create this structure in your runtime:
dataset/train/images, dataset/train/labels, dataset/valid/images, dataset/valid/labels, dataset/test/images, dataset/test/labels


In [4]:
import zipfile
import shutil

# Preferred local training path
local_dataset = Path('dataset')
local_dataset.mkdir(parents=True, exist_ok=True)

# Candidate dataset roots on Drive
drive_dataset_candidates = [
    Path('/content/drive/MyDrive/datasets'),
    Path('/content/drive/MyDrive/dataset'),
]

# If local dataset is missing or partial, sync from Drive dataset folders
for drive_root in drive_dataset_candidates:
    if not drive_root.exists():
        continue

    print(f'Found Drive dataset candidate: {drive_root}')

    # Copy data.yaml if missing
    if not (local_dataset / 'data.yaml').exists() and (drive_root / 'data.yaml').exists():
        shutil.copy2(drive_root / 'data.yaml', local_dataset / 'data.yaml')
        print('Copied data.yaml from Drive.')

    # Copy required split folders only if missing locally
    for split in ['train', 'valid', 'test']:
        src = drive_root / split
        dst = local_dataset / split
        if src.exists() and not dst.exists():
            print(f'Copying missing split: {src} -> {dst}')
            shutil.copytree(src, dst, dirs_exist_ok=True)

# Fall back to archive extraction if still missing data.yaml
archive_candidates = [
    Path('archive.zip'),
    Path('dataset/archive.zip'),
    Path('/content/drive/MyDrive/archive.zip'),
    Path('/content/drive/MyDrive/datasets/archive.zip'),
    Path('/content/drive/MyDrive/dataset/archive.zip'),
]
archive_path = next((p for p in archive_candidates if p.exists()), None)

if not (local_dataset / 'data.yaml').exists() and archive_path is not None:
    print(f'Extracting {archive_path} -> dataset/...')
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(Path('.'))

    # Handle common nested structure: dataset/dataset/...
    nested_yaml = Path('dataset/dataset/data.yaml')
    if nested_yaml.exists() and not (local_dataset / 'data.yaml').exists():
        for item in Path('dataset/dataset').iterdir():
            target = local_dataset / item.name
            if target.exists():
                continue
            shutil.move(str(item), str(target))

# Final status
if (local_dataset / 'data.yaml').exists():
    print('dataset/data.yaml exists.')
else:
    print('dataset/data.yaml is still missing.')

for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        p = local_dataset / split / sub
        print(f"{'OK ' if p.exists() else 'MISS'} - {p}")

# IMPORTANT: force notebook variables to local dataset for speed and consistency
if (local_dataset / 'data.yaml').exists():
    PROJECT_ROOT = Path('.')
    DATASET_ROOT = local_dataset
    DATA_YAML = DATASET_ROOT / 'data.yaml'
    print(f'Using LOCAL dataset path: {DATA_YAML}')

dataset/data.yaml is still missing.
MISS - dataset/train/images
MISS - dataset/train/labels
MISS - dataset/valid/images
MISS - dataset/valid/labels
MISS - dataset/test/images
MISS - dataset/test/labels


## 1.5 Prepare Dataset in Runtime (Optional)

Run this cell once if `dataset/...` folders are missing. It can extract from `archive.zip` if present in current directory.

## 2. Validate Dataset Structure

In [35]:
expected_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

print('Dataset directory check:')
all_ok = True
for p in expected_dirs:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISS'} - {p}")
    all_ok = all_ok and ok

if not all_ok:
    raise FileNotFoundError('One or more required dataset folders are missing. Fix dataset structure first.')

print('All required dataset directories exist.')

Dataset directory check:
MISS - dataset/train/images
MISS - dataset/train/labels
OK  - dataset/valid/images
OK  - dataset/valid/labels
MISS - dataset/test/images
MISS - dataset/test/labels


FileNotFoundError: One or more required dataset folders are missing. Fix dataset structure first.

## 3. Configure Dataset YAML

In [33]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Missing file: {DATA_YAML}')

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

# Auto-normalize common Roboflow exports (../train/images -> train/images)
normalize_map = {
    '../train/images': 'train/images',
    '../valid/images': 'valid/images',
    '../test/images': 'test/images',
}
changed = False
for k in ['train', 'val', 'test']:
    cur = data_cfg.get(k)
    if cur in normalize_map:
        data_cfg[k] = normalize_map[cur]
        changed = True

if changed:
    with open(DATA_YAML, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False)
    print('Updated data.yaml paths to local-relative format.')

print('Loaded data.yaml:')
print(json.dumps({
    'train': data_cfg.get('train'),
    'val': data_cfg.get('val'),
    'test': data_cfg.get('test'),
    'nc': data_cfg.get('nc'),
    'names': data_cfg.get('names'),
}, indent=2))

expected_paths = {
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
}
expected_names = ['handrise', 'read', 'sleep', 'stand', 'using_electronic_devices', 'write']

for k, v in expected_paths.items():
    if data_cfg.get(k) != v:
        raise ValueError(f"data.yaml {k} should be '{v}', got '{data_cfg.get(k)}'")

names = data_cfg.get('names', [])
nc = int(data_cfg.get('nc', -1))
if nc != len(names):
    raise ValueError(f"nc ({nc}) does not match len(names) ({len(names)})")

if list(names) != expected_names:
    print('Warning: class names differ from expected exported labels.')
    print('Expected:', expected_names)
    print('Found   :', names)
else:
    print('Class labels match expected exported labels.')

print('data.yaml validation complete.')

Loaded data.yaml:
{
  "train": "train/images",
  "val": "valid/images",
  "test": "test/images",
  "nc": 6,
  "names": [
    "handrise",
    "read",
    "sleep",
    "stand",
    "using_electronic_devices",
    "write"
  ]
}
Class labels match expected exported labels.
data.yaml validation complete.


## 4. Train YOLOv11 Model

In [ ]:
model = YOLO('yolo11n.pt')

# If dataset exists locally, always train from local path (faster than Drive)
if Path('dataset/data.yaml').exists():
    DATA_YAML = Path('dataset/data.yaml')

# Fast smoke-test mode for Colab (set to False for full training)
QUICK_RUN = True

if QUICK_RUN:
    epochs = 5
    imgsz = 512
    batch = 8
else:
    epochs = 20
    imgsz = 640
    batch = 16

train_config = {
    'data': str(DATA_YAML),
    'epochs': epochs,
    'imgsz': imgsz,
    'batch': batch,
    'device': device,
    'project': 'fyp_runs',
    'name': 'classroom_model_v1',
    'verbose': True,
    'save': True,
    'workers': 0,  # avoid multiprocessing stalls and cache race on Drive/Colab
    'cache': False,
}

print('Training config:')
for k, v in train_config.items():
    print(f'  {k}: {v}')

try:
    train_results = model.train(**train_config)
    print('Training complete.')
except RuntimeError as e:
    msg = str(e).lower()
    if 'out of memory' in msg or 'cudnn' in msg:
        print('OOM/accelerator error detected. Try batch=4 and rerun this cell.')
    raise

Training config:
  data: /content/drive/MyDrive/dataset/data.yaml
  epochs: 20
  imgsz: 640
  batch: 16
  device: 0
  project: fyp_runs
  name: classroom_model_v1
  verbose: True
  save: True
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

KeyboardInterrupt: 

## 5. Run Live Inference on Webcam

Training notebook note: live webcam testing is done in [live_inference.ipynb](live_inference.ipynb). This cell checks whether trained weights exist for that notebook.

In [ ]:
print('Checking weights for live inference...')
if BEST_WEIGHTS.exists():
    size_mb = BEST_WEIGHTS.stat().st_size / (1024 * 1024)
    print(f'Found: {BEST_WEIGHTS} ({size_mb:.2f} MB)')
    print('You can now run live_inference.ipynb.')
else:
    print(f'Not found yet: {BEST_WEIGHTS}')
    print('Run the training cell above first.')

## 6. Visualize Training Results

In [ ]:
results_png = RUN_DIR / 'results.png'
cm_png = RUN_DIR / 'confusion_matrix.png'
val_batch_png = RUN_DIR / 'val_batch0_pred.jpg'

print('Artifacts:')
for p in [results_png, cm_png, val_batch_png]:
    print(f"{'OK ' if p.exists() else 'MISS'} - {p}")

# Show charts if available
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
images = [results_png, cm_png, val_batch_png]
titles = ['Training Curves', 'Confusion Matrix', 'Sample Validation Predictions']

for ax, img_path, title in zip(axes, images, titles):
    ax.set_title(title)
    ax.axis('off')
    if img_path.exists():
        img = Image.open(img_path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'Not generated yet', ha='center', va='center')

plt.tight_layout()
plt.show()